In [ ]:
import cv2
import numpy as np
import os


folders = ["frame","enhanced", "threshold", "morphology", "masks", "output"]
for f in folders:
    os.makedirs(f, exist_ok=True)


def enhance(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    median = cv2.medianBlur(gray, 3)

    log_img = np.log1p(median.astype(np.float32))
    log_img = cv2.normalize(log_img, None, 0, 255, cv2.NORM_MINMAX)
    log_img = np.uint8(log_img)

    enhanced = cv2.equalizeHist(log_img)
    return enhanced


def segment(enhanced, original):
    h_img, w_img = enhanced.shape 
    blur=cv2.GaussianBlur(enhanced, (5,5), 0)
    
    _, thresh = cv2.threshold(enhanced, 0, 255,
                             cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

   
    roi_mask = np.zeros_like(thresh)
    pts = np.array([[(0, h_img), (int(w_img*0.1), int(h_img*0.25)),
                     (int(w_img*0.9), int(h_img*0.25)), (w_img, h_img)]])
    cv2.fillPoly(roi_mask, pts, 255)
    thresh = cv2.bitwise_and(thresh, roi_mask)

    kernel = np.ones((5,5), np.uint8)
    morph = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, 2)
    morph = cv2.morphologyEx(morph, cv2.MORPH_OPEN, kernel, 1)

    contours, _ = cv2.findContours(morph, cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_SIMPLE)

    mask = np.zeros_like(enhanced)

    for cnt in contours:
        if cv2.contourArea(cnt) < 200:
            continue

        x, y, w_box, h_box = cv2.boundingRect(cnt)  
        roi = enhanced[y:y+h_box, x:x+w_box]

        if np.mean(roi) > 200 or np.var(roi) < 25:
            continue

        cv2.drawContours(mask, [cnt], -1, 255, -1)
        cv2.drawContours(original, [cnt], -1, (0,0,255), 2)

    return thresh, morph, mask, original


def evaluate(pred, gt):
    _, pred = cv2.threshold(pred, 127, 1, cv2.THRESH_BINARY)
    _, gt = cv2.threshold(gt, 127, 1, cv2.THRESH_BINARY)

    TP = np.sum((pred == 1) & (gt == 1))
    TN = np.sum((pred == 0) & (gt == 0))
    FP = np.sum((pred == 1) & (gt == 0))
    FN = np.sum((pred == 0) & (gt == 1))

    accuracy = (TP + TN) / (TP + TN + FP + FN + 1e-6)
    precision = TP / (TP + FP + 1e-6)
    recall = TP / (TP + FN + 1e-6)
    f1 = 2 * precision * recall / (precision + recall + 1e-6)
    iou = TP / (TP + FP + FN + 1e-6)
    dice = (2 * TP) / (2 * TP + FP + FN + 1e-6)

    return accuracy, precision, recall, f1, iou, dice


cap = cv2.VideoCapture("final_video.mp4")

frame_id = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    original = frame.copy()
    cv2.imwrite(f"frame/frame_{frame_id}.jpg", original)
   
    enhanced = enhance(frame)
    cv2.imwrite(f"enhanced/frame_{frame_id}.jpg", enhanced)

   
    thresh, morph, mask, output = segment(enhanced, original)

    cv2.imwrite(f"threshold/frame_{frame_id}.jpg", thresh)
    cv2.imwrite(f"morphology/frame_{frame_id}.jpg", morph)
    cv2.imwrite(f"masks/frame_{frame_id}.png", mask)
    cv2.imwrite(f"output/frame_{frame_id}.jpg", output)

    cv2.imshow("Final Output", output)

    if cv2.waitKey(1) & 0xFF == 27:
        break

    frame_id += 1

cap.release()
cv2.destroyAllWindows()

print("Processing done!")


print("\n===== Evaluation =====")

test_frames = [109, 164, 223, 366, 416, 421]

for f in test_frames:
    pred = cv2.imread(f"masks/frame_{f}.png", 0)
    gt = cv2.imread(f"ground_truth/gt_{f}.jpg", 0)

    if pred is None or gt is None:
        print(f"Frame {f}: Missing files")
        continue

    acc, pre, rec, f1, iou, dice = evaluate(pred, gt)

    print(f"\nFrame {f}")
    print("Accuracy :", round(acc, 4))
    print("Precision:", round(pre, 4))
    print("Recall   :", round(rec, 4))
    print("F1 Score :", round(f1, 4))
    print("IoU      :", round(iou, 4))
    print("Dice     :", round(dice, 4))

Processing done!

===== Evaluation =====

Frame 109
Accuracy : 0.9006
Precision: 0.9674
Recall   : 0.8044
F1 Score : 0.8784
IoU      : 0.7831
Dice     : 0.8784

Frame 164
Accuracy : 0.9149
Precision: 0.7927
Recall   : 0.9477
F1 Score : 0.8633
IoU      : 0.7595
Dice     : 0.8633

Frame 223
Accuracy : 0.8804
Precision: 0.7279
Recall   : 0.8085
F1 Score : 0.7661
IoU      : 0.6208
Dice     : 0.7661

Frame 366
Accuracy : 0.8401
Precision: 0.6979
Recall   : 0.9559
F1 Score : 0.8068
IoU      : 0.6761
Dice     : 0.8068

Frame 416
Accuracy : 0.837
Precision: 0.7113
Recall   : 0.9759
F1 Score : 0.8228
IoU      : 0.699
Dice     : 0.8228

Frame 421
Accuracy : 0.9323
Precision: 0.9693
Recall   : 0.9195
F1 Score : 0.9438
IoU      : 0.8935
Dice     : 0.9438
